# 📧 Spam Email Classifier using Machine Learning
**Author:** Shrey Dukare  
**College:** BSIOTR, Pune — B.E. Computer Engineering (Final Year)  
**Tools:** Python, Scikit-learn, Pandas, Matplotlib, Seaborn

---

## 🎯 Project Objective
Build a binary text classification model that automatically detects whether an email is **spam** or **ham (not spam)** using Natural Language Processing (NLP) and Machine Learning techniques.

## 📋 Pipeline Overview
1. Load & Explore Dataset
2. Data Preprocessing & Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Extraction using TF-IDF
5. Train Multiple ML Models
6. Evaluate & Compare Models
7. Test on Custom Input
8. Conclusion

---
## Step 1: Install & Import Libraries

In [ ]:
# Install any missing libraries
!pip install scikit-learn pandas matplotlib seaborn wordcloud --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# NLP & ML
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

---
## Step 2: Load Dataset
We use the **SMS Spam Collection Dataset** — a classic NLP benchmark dataset with 5,574 messages labelled as spam or ham.

In [ ]:
# Load directly from UCI ML Repository
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print(f'Dataset Shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

---
## Step 3: Basic Exploration

In [ ]:
# Check label distribution
print('Label Distribution:')
print(df['label'].value_counts())
print(f'\nSpam %: {df["label"].value_counts(normalize=True)["spam"]*100:.2f}%')
print(f'Ham  %: {df["label"].value_counts(normalize=True)["ham"]*100:.2f}%')

# Check for missing values
print(f'\nMissing Values:\n{df.isnull().sum()}')

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', rot=0)
axes[0].set_title('Spam vs Ham — Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, v in enumerate(df['label'].value_counts()):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                 colors=colors, startangle=90, explode=[0, 0.05])
axes[1].set_title('Spam vs Ham — Proportion', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Dataset is imbalanced — more ham than spam (realistic!)')

---
## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Add message length feature
df['msg_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda x: len(x.split()))

print('Average message length by label:')
print(df.groupby('label')[['msg_length', 'word_count']].mean().round(2))

In [ ]:
# Message length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in zip(['ham', 'spam'], ['#2ecc71', '#e74c3c']):
    axes[0].hist(df[df['label'] == label]['msg_length'],
                 bins=40, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0].set_title('Message Length Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot
df.boxplot(column='msg_length', by='label', ax=axes[1],
           boxprops=dict(color='navy'), medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Message Length by Label', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Character Count')
plt.suptitle('')

plt.tight_layout()
plt.savefig('length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Spam messages tend to be longer than ham messages!')

In [ ]:
# Word Cloud for SPAM messages
spam_words = ' '.join(df[df['label'] == 'spam']['message'].tolist())
ham_words  = ' '.join(df[df['label'] == 'ham']['message'].tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

wc_spam = WordCloud(width=700, height=400, background_color='white',
                    colormap='Reds', max_words=100).generate(spam_words)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('🚨 SPAM — Most Common Words', fontsize=14, fontweight='bold', color='red')

wc_ham = WordCloud(width=700, height=400, background_color='white',
                   colormap='Greens', max_words=100).generate(ham_words)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('✅ HAM — Most Common Words', fontsize=14, fontweight='bold', color='green')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Spam messages contain words like FREE, CALL, WIN, CLAIM — classic red flags!')

---
## Step 5: Text Preprocessing

Raw text needs to be cleaned before we can feed it to ML models:
- Convert to **lowercase**
- Remove **punctuation** and **special characters**
- Remove **extra whitespace**
- (Optional) Remove **stop words** like 'the', 'is', 'at'

In [ ]:
def preprocess_text(text):
    """
    Clean and normalize raw text:
    1. Lowercase
    2. Remove URLs
    3. Remove punctuation & special characters
    4. Remove extra whitespace
    """
    text = text.lower()                                  # Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)          # Remove URLs
    text = re.sub(r'\d+', '', text)                      # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()            # Remove extra spaces
    return text

# Apply preprocessing
df['clean_message'] = df['message'].apply(preprocess_text)

# Preview
print('Before cleaning:')
print(df['message'].iloc[0])
print('\nAfter cleaning:')
print(df['clean_message'].iloc[0])

---
## Step 6: Feature Extraction using TF-IDF

**TF-IDF** (Term Frequency - Inverse Document Frequency) converts text into numerical vectors.
- **TF**: How often a word appears in this message
- **IDF**: Penalizes words that appear in ALL messages (like 'the', 'is') — giving more weight to unique/important words

This is the core NLP step that lets ML models understand text.

In [ ]:
# Encode labels: spam=1, ham=0
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X = df['clean_message']
y = df['label_num']

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,    # Keep top 5000 words
    stop_words='english', # Remove common English stop words
    ngram_range=(1, 2)    # Use unigrams AND bigrams (e.g., 'free prize')
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'\nTF-IDF Matrix Shape (train): {X_train_tfidf.shape}')
print('Each message is now represented as a vector of 5000 features!')

---
## Step 7: Train & Compare 3 ML Models

| Model | Why use it? |
|---|---|
| **Naive Bayes** | Fast, works great for text, classic spam filter algorithm |
| **Logistic Regression** | Strong baseline, interpretable, handles high-dimensional data |
| **LinearSVC** | High-performance linear classifier, great for text |

In [ ]:
# Define models
models = {
    'Naive Bayes'         : MultinomialNB(),
    'Logistic Regression' : LogisticRegression(max_iter=1000, C=1.0),
    'LinearSVC'           : LinearSVC(max_iter=1000)
}

results = {}

print('=' * 60)
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'predictions': y_pred, 'accuracy': acc}
    print(f'\n🔷 {name}')
    print(f'   Accuracy: {acc*100:.2f}%')
    print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))
    print('-' * 60)

best_model_name = max(results, key=lambda k: results[k]['accuracy'])
print(f'\n🏆 Best Model: {best_model_name} ({results[best_model_name]["accuracy"]*100:.2f}% accuracy)')

---
## Step 8: Visualize Results

In [ ]:
# Accuracy comparison bar chart
names = list(results.keys())
accs  = [results[n]['accuracy'] * 100 for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(names, accs, color=['#3498db', '#e67e22', '#9b59b6'],
                   edgecolor='black', width=0.5)
axes[0].set_ylim(90, 100)
axes[0].set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy (%)')
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=11)

# Confusion matrix for best model
best_preds = results[best_model_name]['predictions']
cm = confusion_matrix(y_test, best_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\n📊 Confusion Matrix Breakdown ({best_model_name}):')
print(f'   True Negatives  (Ham correctly identified) : {tn}')
print(f'   False Positives (Ham wrongly flagged spam) : {fp}')
print(f'   False Negatives (Spam missed)              : {fn}')
print(f'   True Positives  (Spam correctly caught)    : {tp}')

In [ ]:
# Top spam-indicating words (Logistic Regression coefficients)
lr_model = results['Logistic Regression']['model']
feature_names = tfidf.get_feature_names_out()
coefs = lr_model.coef_[0]

top_spam_idx = coefs.argsort()[-20:][::-1]
top_ham_idx  = coefs.argsort()[:20]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh([feature_names[i] for i in top_spam_idx],
             [coefs[i] for i in top_spam_idx], color='#e74c3c')
axes[0].set_title('Top 20 SPAM Indicator Words', fontweight='bold', color='red')
axes[0].set_xlabel('Coefficient Weight')

axes[1].barh([feature_names[i] for i in top_ham_idx],
             [coefs[i] for i in top_ham_idx], color='#2ecc71')
axes[1].set_title('Top 20 HAM Indicator Words', fontweight='bold', color='green')
axes[1].set_xlabel('Coefficient Weight')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9: Test on Your Own Messages!
Try the classifier with custom messages and see what it predicts.

In [ ]:
def predict_spam(message, model_name='Logistic Regression'):
    """
    Predict if a message is spam or ham.
    Uses the trained TF-IDF vectorizer + selected model.
    """
    cleaned = preprocess_text(message)
    vectorized = tfidf.transform([cleaned])
    model = results[model_name]['model']
    prediction = model.predict(vectorized)[0]
    label = '🚨 SPAM' if prediction == 1 else '✅ HAM'
    print(f'Message  : "{message}"')
    print(f'Prediction: {label}\n')

# --- Test with sample messages ---
test_messages = [
    "Congratulations! You've won a FREE iPhone. Click here to claim your prize NOW!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your bank account has been suspended. Call 1800-XXX-XXXX immediately.",
    "Don't forget to submit your assignment before Friday.",
    "Win a cash prize of Rs 50,000! Reply WIN to 56161 now. Limited offer!",
    "Mom, I'll be home by 8pm tonight."
]

print('=' * 60)
print('       SPAM CLASSIFIER — CUSTOM TEST RESULTS')
print('=' * 60)
for msg in test_messages:
    predict_spam(msg)

In [ ]:
# ✏️ TRY YOUR OWN MESSAGE HERE!
my_message = "Type your message here and run this cell!"
predict_spam(my_message)

---
## Step 10: Summary & Conclusion

### ✅ What We Built
A complete end-to-end **Spam Email Classifier** that:
- Loads and explores real-world SMS data (5,574 samples)
- Cleans and preprocesses text using NLP techniques
- Extracts features using **TF-IDF** (with bigrams)
- Trains and compares **3 ML models**: Naive Bayes, Logistic Regression, LinearSVC
- Achieves **~97–98% accuracy** on test data
- Identifies key spam-indicator words via feature importance
- Predicts on custom real-world messages

### 📌 Key Learnings
| Concept | What We Learned |
|---|---|
| TF-IDF | Converts text into meaningful numerical features |
| Naive Bayes | Probabilistic classifier, works well for text |
| Logistic Regression | Strong baseline, interpretable coefficients |
| Precision vs Recall | For spam: high recall = catch more spam, high precision = fewer false alarms |
| Confusion Matrix | Visualizes TP, TN, FP, FN — critical for imbalanced datasets |

### 🔮 Possible Improvements
1. Use **NLTK stemming/lemmatization** for better text normalization
2. Try **BERT / DistilBERT** (transformer-based) for even higher accuracy
3. Build a **web interface** using Flask or Streamlit
4. Train on **email subject + body** for a more realistic classifier
5. Handle **class imbalance** using SMOTE or class_weight='balanced'

---
**Author:** Shrey Dukare | BSIOTR, Pune | shreydokre@gmail.com